# Trustworthy CXR VLM — external localization test on NIH ChestX-ray14 radiologist boxes (DR-034)

Scores the IU-trained S2 Grad-CAM head against the 984 radiologist-drawn boxes of NIH ChestX-ray14
(`BBox_List_2017.csv`, CC0). Nothing is tuned. **CPU is enough** (no GPU quota needed).

**Before running:** Accelerator **None (CPU)**, Internet **On**, inputs: **NIH Chest X-rays** (nih-chest-xrays/data)
and your **xai-head** dataset (the uploaded `xai_head.pt`). Download `nih_external_results.zip` when done.

In [ ]:
import glob, os, subprocess, sys
REPO_URL = "https://github.com/justinthomas11/mts-medical-vlm.git"
WORK = "/kaggle/working/mts-medical-vlm"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
os.chdir(WORK)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q "transformers==5.17.0" accelerate
BBOX = glob.glob("/kaggle/input/**/BBox_List_2017.csv", recursive=True)[0]
NIH_ROOT = os.path.dirname(BBOX)
HEAD = glob.glob("/kaggle/input/**/xai_head.pt", recursive=True)[0]
print("NIH:", NIH_ROOT, "| head:", HEAD)
os.makedirs("checkpoints", exist_ok=True)
subprocess.run(["cp", HEAD, "checkpoints/xai_head.pt"], check=True)
# Link the boxed images where configs/data_config.yaml (paths.nih_bbox_dir) expects them.
import csv
dst = "data/external/nih_cxr14"
os.makedirs(f"{dst}/images", exist_ok=True)
if not os.path.exists(f"{dst}/BBox_List_2017.csv"):
    os.symlink(BBOX, f"{dst}/BBox_List_2017.csv")
want = {r["Image Index"] for r in csv.DictReader(open(BBOX))}
found = 0
for p in glob.glob(f"{NIH_ROOT}/images_*/images/*.png"):
    n = os.path.basename(p)
    if n in want:
        if not os.path.exists(f"{dst}/images/{n}"):
            os.symlink(p, f"{dst}/images/{n}")
        found += 1
print(found, "of", len(want), "boxed images linked")
assert found == len(want), "some boxed images were not found in the NIH input"

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_external_localization.py", "-q"], capture_output=True, text=True)
print(r.stdout[-800:])
subprocess.run([sys.executable, "src/pipeline/eval_external_localization.py"], check=True)

In [ ]:
import zipfile
out = "/kaggle/working/nih_external_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob("results/external/*")):
        z.write(f)
    z.writestr("results/external/KAGGLE_RUN.txt",
               subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
print(out)